In [0]:
from pyspark.sql import SparkSession

In [0]:
#CONFIGURACION DE CATALOGO, ESQUEMAS Y RUTAS
PROYECTO        = "ventas_retail_jorgemendoza"
CATALOGO        = "proyecto_final"
ESQUEMA_LANDING         = "landing"
ESQUEMA_BRONZE         = "bronze"
ESQUEMA_SILVER         = "silver"
ESQUEMA_GOLD           = "gold"
NOMBRE_VOLUME           = "raw_data"
ENTIDAD_CLIENTE        = "clientes"
ENTIDAD_PRODUCTO       = "productos"
ENTIDAD_PEDIDO          = "pedidos"
ENTIDAD_DETALLE         = "detalle_pedidos"


In [0]:
#CREACION DEL CATALOGO
spark.sql(f"""CREATE CATALOG IF NOT EXISTS {CATALOGO}""")
spark.sql("SHOW CATALOGS").show()
spark.sql(f"USE CATALOG {CATALOGO}")
print(f"Usando catálogo: {CATALOGO}")

In [0]:
#CREACION DE LOS SCHEMAS
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{ESQUEMA_LANDING}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{ESQUEMA_BRONZE}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{ESQUEMA_SILVER}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{ESQUEMA_GOLD}")
spark.sql("SHOW SCHEMAS").show()

In [0]:
#CREAR VOLUME 
spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA_LANDING}.{NOMBRE_VOLUME}
""")
print(NOMBRE_VOLUME)

In [0]:
#CREAR CARPETA
ruta = f"/Volumes/{CATALOGO}/{ESQUEMA_LANDING}/{NOMBRE_VOLUME}/{PROYECTO}"

try:
    dbutils.fs.ls(ruta)
    print(f"La carpeta ya existe: {ruta}")
except Exception:
    dbutils.fs.mkdirs(ruta)
    print(f"Carpeta creada: {ruta}")

In [0]:
#CREAR SUB-CARPETAS
ruta_base = f"/Volumes/{CATALOGO}/{ESQUEMA_LANDING}/{NOMBRE_VOLUME}/{PROYECTO}"

entidades = [ENTIDAD_CLIENTE,ENTIDAD_PRODUCTO,ENTIDAD_PEDIDO,ENTIDAD_DETALLE]

for entidad in entidades:
    ruta_entidad = f"{ruta_base}/{entidad}"
    try:
        dbutils.fs.ls(ruta_entidad)
        print(f"La carpeta ya existe: {ruta_entidad}")
    except Exception:
        dbutils.fs.mkdirs(ruta_entidad)
        print(f"Carpeta creada: {ruta_entidad}")

In [0]:

# VERIFICAR ARCHIVOS
import os

ruta_clientes = f"{ruta_base}/{ENTIDAD_CLIENTE}"
ruta_productos = f"{ruta_base}/{ENTIDAD_PRODUCTO}"
ruta_pedidos = f"{ruta_base}/{ENTIDAD_PEDIDO}"
ruta_detalle = f"{ruta_base}/{ENTIDAD_DETALLE}"

archivos_esperados = [
    f"{ruta_clientes}/clientes_batch_1.csv",
    f"{ruta_clientes}/clientes_batch_2.csv",
    f"{ruta_clientes}/clientes_batch_3.csv",
    f"{ruta_productos}/productos_batch_1.csv",
    f"{ruta_productos}/productos_batch_2.csv",
    f"{ruta_productos}/productos_batch_3.csv",
    f"{ruta_pedidos}/pedidos_batch_1.json",
    f"{ruta_pedidos}/pedidos_batch_2.json",
    f"{ruta_pedidos}/pedidos_batch_3.json",
    f"{ruta_detalle}/detalle_pedidos_batch_1.json",
    f"{ruta_detalle}/detalle_pedidos_batch_2.json",
    f"{ruta_detalle}/detalle_pedidos_batch_3.json",  
]

todos_presentes = True
for ruta in archivos_esperados:
    existe = os.path.exists(ruta)
    estado = "OK" if existe else "FALTA"
    print(f"[{estado}] {ruta}")
    if not existe:
        todos_presentes = False

if not todos_presentes:
    raise FileNotFoundError(
        "Uno o más archivos fuente no están disponibles. "
        "Subir los CSV al Volume antes de continuar."
    )

print("\nTodos los archivos fuente verificados.")
